# Phase 2 - Notebook 00: 3DGS + SLAM Overview

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase2/00_phase2_overview.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand why combining 3DGS with SLAM is powerful
2. Learn the key challenges and solutions
3. Know the main approaches (SplaTAM, MonoGS, GS-SLAM)
4. Understand the phase 2 learning path

**Estimated Time**: 45 minutes

**Prerequisites**: Phase 1 (Notebooks 00-10)

---

## 1. Introduction

### What is SLAM?

**SLAM (Simultaneous Localization and Mapping)** solves the chicken-and-egg problem:
- To know where you are, you need a map
- To build a map, you need to know where you are

SLAM does both **simultaneously** from sensor data (camera, LiDAR, etc.).

### Why 3DGS + SLAM?

| Traditional SLAM | NeRF-SLAM | 3DGS-SLAM |
|------------------|-----------|------------|
| Sparse points/features | Dense but slow | Dense AND fast |
| Real-time capable | Not real-time | Real-time capable |
| No rendering | Novel view synthesis | Novel view synthesis |
| Limited appearance | Photorealistic | Photorealistic |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Traditional SLAM
ax = axes[0]
np.random.seed(42)
points = np.random.randn(50, 2) * 2
ax.scatter(points[:, 0], points[:, 1], s=30, c='blue', alpha=0.6)
# Camera trajectory
t = np.linspace(0, 2*np.pi, 20)
traj_x = np.cos(t) * 3
traj_y = np.sin(t) * 3
ax.plot(traj_x, traj_y, 'r-', linewidth=2)
ax.scatter(traj_x[::3], traj_y[::3], s=100, c='red', marker='^')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_title('Traditional SLAM\n(Sparse Points + Poses)')
ax.set_aspect('equal')

# NeRF-SLAM
ax = axes[1]
# Dense grid representing neural field
x = np.linspace(-4, 4, 30)
y = np.linspace(-4, 4, 30)
X, Y = np.meshgrid(x, y)
Z = np.sin(X) * np.cos(Y) + np.random.randn(*X.shape) * 0.1
ax.contourf(X, Y, Z, levels=20, cmap='viridis', alpha=0.7)
ax.plot(traj_x, traj_y, 'r-', linewidth=2)
ax.scatter(traj_x[::3], traj_y[::3], s=100, c='red', marker='^')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_title('NeRF-SLAM\n(Dense but Slow)')
ax.set_aspect('equal')

# 3DGS-SLAM
ax = axes[2]
from matplotlib.patches import Ellipse
for _ in range(100):
    cx, cy = np.random.randn(2) * 2
    rx, ry = np.random.rand(2) * 0.5 + 0.1
    angle = np.random.rand() * 360
    color = plt.cm.viridis(np.random.rand())
    ellipse = Ellipse((cx, cy), rx, ry, angle=angle, 
                     fill=True, alpha=0.3, facecolor=color)
    ax.add_patch(ellipse)
ax.plot(traj_x, traj_y, 'r-', linewidth=2)
ax.scatter(traj_x[::3], traj_y[::3], s=100, c='red', marker='^')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_title('3DGS-SLAM\n(Dense AND Fast)')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 2. The 3DGS-SLAM Paradigm

### Core Idea

Replace traditional map representations with **3D Gaussians**:

```
┌─────────────────────────────────────────────────────────────┐
│                    3DGS-SLAM Pipeline                        │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│   RGB(-D)       ┌──────────┐      ┌──────────┐              │
│   Input    ──►  │ Tracking │  ──► │ Mapping  │              │
│                 └────┬─────┘      └────┬─────┘              │
│                      │                 │                     │
│                      ▼                 ▼                     │
│                 Camera Pose      Gaussian Map                │
│                      │                 │                     │
│                      └────────┬────────┘                     │
│                               │                              │
│                               ▼                              │
│                    ┌────────────────────┐                    │
│                    │  Render & Compare  │                    │
│                    │   (Photometric)    │                    │
│                    └────────────────────┘                    │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

### Key Components

| Component | Function | How 3DGS Helps |
|-----------|----------|----------------|
| **Tracking** | Estimate camera pose | Render-and-compare |
| **Mapping** | Build/update map | Gaussian optimization |
| **Loop Closure** | Correct drift | Re-rendering verification |
| **Visualization** | View reconstruction | Real-time rendering |

## 3. Key Approaches

### SplaTAM (CVPR 2024)

**"Splat, Track & Map 3D Gaussians for Dense RGB-D SLAM"**

- Input: RGB-D (requires depth)
- Tracking: Gradient-based pose optimization
- Mapping: Silhouette-guided densification
- Key innovation: Joint optimization of camera and Gaussians

### MonoGS (CVPR 2024)

**"Gaussian Splatting SLAM"**

- Input: Monocular RGB (no depth required)
- Tracking: Photometric alignment
- Mapping: Geometry regularization
- Key innovation: Works without depth sensor

### GS-SLAM

**"Dense Visual SLAM with 3D Gaussian Splatting"**

- Similar to SplaTAM
- Focus on efficiency
- Adaptive Gaussian management

In [ ]:
# Method comparison
comparison = {
    'Method': ['SplaTAM', 'MonoGS', 'GS-SLAM', 'ORB-SLAM3', 'NICE-SLAM'],
    'Input': ['RGB-D', 'Mono/Stereo', 'RGB-D', 'Mono/Stereo/RGB-D', 'RGB-D'],
    'Representation': ['3D Gaussians', '3D Gaussians', '3D Gaussians', 'Points + BoW', 'Neural Grid'],
    'Real-time': ['Yes', 'Near', 'Yes', 'Yes', 'No'],
    'Novel View': ['Yes', 'Yes', 'Yes', 'No', 'Yes'],
}

print("Method Comparison:")
print("=" * 80)
header = f"{'Method':12} | {'Input':15} | {'Representation':15} | {'RT':5} | {'NVS':5}"
print(header)
print("-" * 80)
for i in range(len(comparison['Method'])):
    row = f"{comparison['Method'][i]:12} | {comparison['Input'][i]:15} | {comparison['Representation'][i]:15} | {comparison['Real-time'][i]:5} | {comparison['Novel View'][i]:5}"
    print(row)

## 4. Key Challenges

### Challenge 1: Online Learning

Unlike offline 3DGS training:
- Cannot iterate over all views
- Must process frames sequentially
- Need to balance old vs new observations

**Solution**: Keyframe-based optimization, sliding window

### Challenge 2: Scale Ambiguity (Monocular)

Monocular cameras cannot determine absolute scale:
- Small nearby object looks same as large distant object
- Affects Gaussian scales and positions

**Solution**: Regularization, depth priors, or use RGB-D

### Challenge 3: Catastrophic Forgetting

Optimizing for new views can degrade old views:
- Gaussians drift from original positions
- Appearance changes

**Solution**: Keyframe replay, local optimization windows

### Challenge 4: Initialization

Where do initial Gaussians come from?
- No COLMAP point cloud available
- Must initialize from depth or structure-from-motion

**Solution**: Depth-based initialization, MVS, learned depth

## 5. Phase 2 Learning Path

### Notebook Structure

| # | Topic | Key Concepts |
|---|-------|-------------|
| 00 | Overview | This notebook |
| 01 | SLAM Basics | VO, BA, loop closure |
| 02 | SplaTAM Architecture | Tracking + Mapping |
| 03 | Map Initialization | Depth-guided Gaussians |
| 04 | Camera Tracking | Render-and-compare |
| 05 | Online Optimization | Incremental training |
| 06 | Keyframe Management | Selection, culling |
| 07 | SplaTAM Walkthrough | Code analysis |
| 08 | MonoGS Comparison | Monocular approach |
| 09 | Custom Dataset | Your own data |

In [ ]:
# Visualize the SLAM loop
slam_loop = """
┌─────────────────────────────────────────────────────────────────────┐
│                    3DGS-SLAM Main Loop                               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   for each frame:                                                    │
│                                                                      │
│   1. TRACKING                                                        │
│      ├── Render Gaussians from previous pose                        │
│      ├── Compare with current image                                 │
│      ├── Compute photometric loss                                   │
│      └── Optimize pose (gradient descent)                           │
│                                                                      │
│   2. KEYFRAME DECISION                                               │
│      ├── Check motion threshold                                     │
│      ├── Check overlap with existing keyframes                      │
│      └── If passes: add as keyframe                                 │
│                                                                      │
│   3. MAPPING (if keyframe)                                           │
│      ├── Add new Gaussians from depth                               │
│      ├── Select nearby keyframes                                    │
│      ├── Jointly optimize Gaussians                                 │
│      └── Densify/Prune as needed                                    │
│                                                                      │
│   4. GLOBAL OPTIMIZATION (periodically)                              │
│      ├── Bundle adjustment over all keyframes                       │
│      └── Loop closure detection/correction                          │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
"""
print(slam_loop)

## 6. Getting Started

### Prerequisites

```bash
# Clone SplaTAM
git clone https://github.com/spla-tam/SplaTAM.git --recursive
cd SplaTAM

# Create environment
conda create -n splatam python=3.10
conda activate splatam

# Install dependencies
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
pip install -r requirements.txt

# Install submodules
pip install submodules/diff-gaussian-rasterization
pip install submodules/simple-knn
```

### Datasets

1. **Replica** (recommended for testing)
   - Synthetic, perfect depth
   - Download: See SplaTAM README

2. **TUM RGB-D**
   - Real data, Kinect sensor
   - https://vision.in.tum.de/data/datasets/rgbd-dataset

## 7. Summary

### Key Takeaways

1. **3DGS + SLAM** enables real-time dense reconstruction with novel view synthesis
2. **Main approaches**: SplaTAM (RGB-D), MonoGS (monocular)
3. **Key challenges**: Online learning, scale, forgetting, initialization
4. **Core loop**: Track → Keyframe → Map → Repeat

### What's Next?

In the next notebook, we'll review SLAM fundamentals:

**[01_slam_basics.ipynb](./01_slam_basics.ipynb)** - Visual Odometry, Bundle Adjustment, Loop Closure

---

## References

1. SplaTAM: https://spla-tam.github.io/
2. MonoGS: https://rmurai.co.uk/projects/GaussianSplattingSLAM/
3. GS-SLAM: https://gs-slam.github.io/
4. 3D Gaussian Splatting: https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/